# Ranking pages for refresh review: a decision-support model for search content opportunity

## Abstract

This capstone asks which pages a content team should review first when some pages still receive demand but show signs of decline. Using the anonymized FlyRank refresh dataset, we train a model to score pages for refresh opportunity and compare it with a transparent rule-based baseline. The method uses a client-aware holdout split and a decline label built from observed trend direction, keeping the target grounded in measured outcomes rather than a hand-coded target. The random forest model achieves a Precision@50 of 0.68 versus 0.24 for the baseline, showing a meaningful lift for ranked review prioritization. We present the results as decision-support for editorial triage, not as an automated system for content removal or causal attribution.


## 1. Question

This project asks: which pages still show measurable demand but are likely to need editorial refresh before they continue to drift? The decision it supports is a content-review triage decision: rank candidate pages so a content team can spend limited review time on the highest-value refresh opportunities first.

The work follows a decision-support framing rather than a causal claim. It is not trying to prove Google’s algorithm or claim that refreshing a page causes a specific ranking outcome; it is trying to surface the pages that deserve human review based on observed content and engagement signals.


In [1]:
from pathlib import Path


def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()
question = (
    "Which pages still attract visible demand but have enough decline signal to deserve "
    "editorial refresh review before they lose traction?"
)
print(question)
print("Decision: rank pages for human review, not automate replacement or deletion.")
print(f"Repo root present: {REPO_ROOT.exists()} -> {REPO_ROOT}")


Which pages still attract visible demand but have enough decline signal to deserve editorial refresh review before they lose traction?
Decision: rank pages for human review, not automate replacement or deletion.
Repo root present: True -> /home/farhankabirsifat/Desktop/flyrank-ml-internship_2


## 2. Data

This analysis uses the bundled anonymized starter dataset in `data/raw/content_refresh_anonymized.csv`, which is a public-safe slice of FlyRank content-refresh data. It contains one row per pseudonymized content page and covers a trailing 90-day view of content performance across 32 clients.

The relevant inputs are the structured content metadata and the observed 90-day signals: impressions, clicks, sessions, engagement, scroll behavior, recency, content age, and keyword/search context. The project intentionally excludes client names, domains, page URLs, and raw private queries. We also avoid using the label source columns as features; the target is derived from `trend_direction`, which means the model never learns from the same signal it is trying to predict.

The repo’s processing step filters to pages with impressions and content age above a practical minimum, and then creates a modeled target defined as `trend_direction == "down"`.


In [2]:
from pathlib import Path
import pandas as pd

raw_path = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(raw_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Clients: {df['client_id'].nunique():,}")
print(f"Label positive share: {df['trend_direction'].str.lower().eq('down').mean():.3f}")
print("Protected fields intentionally excluded from public-safe analysis: client names, URLs, domains, keywords, and private queries.")


Rows: 30,000
Columns: 44
Clients: 32
Label positive share: 0.542
Protected fields intentionally excluded from public-safe analysis: client names, URLs, domains, keywords, and private queries.


## 3. Methodology

The target is a binary label for refresh risk: whether a page is in a declining state based on observed recent trend behavior. This is deliberately encoded in the repo as `is_declining_label`, where the label is set to 1 when `trend_direction == "down"`. That keeps the target defined from observed data rather than from a manually imposed rule that would simply recreate the label.

The feature set includes both numeric and categorical variables for visibility, recency, position, depth, engagement, and content characteristics. The preprocessing step fills missing numeric values safely and handles category gaps with explicit unknown values so the model is not silently encoding content-type leakage through a broad fillna pattern.

The baseline is a transparent hand-written refresh score built from visibility, freshness risk, position opportunity, and depth gap. The machine learning comparison then uses a client-aware holdout split, so rows from the same client are held out together rather than mixing clients across train and test sets. This makes the evaluation more realistic for editorial prioritization across multi-page content portfolios.

Leakage checks are built into the workflow: the label source columns are excluded from the model features, and a client-aware split is used instead of a random row-level split. This is aligned with the instruction set in the repo and avoids overstating model performance.


In [3]:
import json
from pathlib import Path

meta_path = REPO_ROOT / "data" / "processed" / "feature_metadata.json"
meta = json.loads(meta_path.read_text())
print("Target definition:", meta["target_definition"])
print("Prepared rows:", meta["prepared_rows"])
print("Declining rate:", round(meta["declining_rate"], 3))
print("Numeric features:", len(meta["model_numeric_features"]))
print("Categorical features:", len(meta["model_categorical_features"]))
print("Top numeric features used by the model:")
for feature in meta["model_numeric_features"][:8]:
    print(" -", feature)


Target definition: trend_direction == 'down'
Prepared rows: 30000
Declining rate: 0.542
Numeric features: 18
Categorical features: 8
Top numeric features used by the model:
 - search_volume
 - competition
 - cpc
 - word_count
 - char_count
 - log_impressions_90d
 - log_clicks_90d
 - log_sessions_90d


## 4. Results (vs baseline)

The best model in this repo is the random forest, selected by Precision@50 on a client-aware holdout split. That is the right metric for this task because the action is ranking, not simply classifying pages as good or bad. A content team does not need perfect labeling of every page; it needs a short, high-quality priority queue.

On the same test split, the baseline rule achieves Precision@50 = 0.24, while the random forest reaches Precision@50 = 0.68. The lift is meaningful: the model is substantially better at surfacing pages that deserve human review. This is a strong decision-support result for triage workflows, while still recognizing that it should support editorial judgment rather than replace it.


In [4]:
import json
from pathlib import Path

results_path = REPO_ROOT / "outputs" / "model_results.json"
results = json.loads(results_path.read_text())
base = results["baseline"]
models = results["models"]
best_name = results["best_model"]["name"]
best = models[best_name]

print(f"Best model selected by Precision@50: {best_name}")
print(f"Baseline Precision@50: {base['baseline_precision_at_50']:.2f}")
print(f"Best model Precision@50: {best['precision_at_50']:.2f}")
print(f"Random forest ROC AUC: {best['roc_auc']:.3f}")
print(f"Random forest Avg Precision: {best['average_precision']:.3f}")
print("\nMetric table:")
for name, metrics in models.items():
    print(name, {
        "precision_at_20": round(metrics["precision_at_20"], 2),
        "precision_at_50": round(metrics["precision_at_50"], 2),
        "precision_at_100": round(metrics["precision_at_100"], 2),
        "roc_auc": round(metrics["roc_auc"], 3),
        "f1": round(metrics["f1"], 3),
    })
print("Baseline", {
    "precision_at_20": round(base["baseline_precision_at_20"], 2),
    "precision_at_50": round(base["baseline_precision_at_50"], 2),
    "precision_at_100": round(base["baseline_precision_at_100"], 2),
    "roc_auc": round(base["baseline_roc_auc"], 3),
})


Best model selected by Precision@50: random_forest
Baseline Precision@50: 0.24
Best model Precision@50: 0.68
Random forest ROC AUC: 0.747
Random forest Avg Precision: 0.610

Metric table:
decision_tree {'precision_at_20': 0.65, 'precision_at_50': 0.66, 'precision_at_100': 0.68, 'roc_auc': 0.742, 'f1': 0.634}
logistic_regression {'precision_at_20': 0.35, 'precision_at_50': 0.4, 'precision_at_100': 0.44, 'roc_auc': 0.7, 'f1': 0.566}
random_forest {'precision_at_20': 0.7, 'precision_at_50': 0.68, 'precision_at_100': 0.7, 'roc_auc': 0.747, 'f1': 0.638}
Baseline {'precision_at_20': 0.15, 'precision_at_50': 0.24, 'precision_at_100': 0.36, 'roc_auc': 0.627}


## 5. Limitations

This project is intentionally modest in its claims. It does not assert that a page’s score causes a specific ranking change, and it does not use client names or domains to claim any external causal effect. The model is designed for editorial prioritization on observed data, not for fully automated content decisions.

The dataset is an anonymized teaching slice and not a full production search-engine audit. Because it captures a trailing 90-day window and uses a content portfolio framing, the results should be interpreted as directional evidence. This means the work is strongest as a decision-support tool for content review, workflow prioritization, and review selection rather than as a full explanation of search ranking dynamics.


In [5]:
print("Honest framing checklist:")
print("- observed: yes")
print("- measured: yes")
print("- directional: yes")
print("- decision-support: yes")
print("- causal ranking claim: no")
print("- client-identifying content: no")


Honest framing checklist:
- observed: yes
- measured: yes
- directional: yes
- decision-support: yes
- causal ranking claim: no
- client-identifying content: no


## 6. Ranked recommendations

The most useful output of this project is not a single prediction; it is a ranked action queue. The model identifies content that still receives visible demand but appears to be losing useful reach, engagement, or freshness. Editorial teams can turn that into an action list such as refresh, expand, improve CTR, or monitor.

Recommended actions in priority order:
1. Refresh pages with strong visible demand but weak recent engagement and declining trend. These are the best candidates for a quick improvement pass.
2. Review low-CTR pages that remain highly visible; they often have ranking presence but weak click efficiency and can be improved with metadata or content repositioning.
3. Expand or modernize thin pages that have search demand but insufficient depth or recency.
4. Monitor pages with moderate risk and stable demand rather than forcing an immediate update.
5. Keep human review in the loop for high-visibility or brand-sensitive content; the model should support judgment, not replace it.


In [7]:
from pathlib import Path
import pandas as pd


def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "outputs" / "refresh_queue.csv").exists():
            return candidate
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    return start


repo_root = globals().get("REPO_ROOT", find_repo_root())
queue_path = repo_root / "outputs" / "refresh_queue.csv"
queue = pd.read_csv(queue_path)
print("Top 10 ranked refresh recommendations:")
print(queue[["final_rank", "final_refresh_score", "suggested_action", "final_reason_codes", "impressions_90d", "sessions_90d", "trend_direction"]].head(10).to_string(index=False))
print("\nAction mix:")
print(queue["suggested_action"].value_counts().to_string())


Top 10 ranked refresh recommendations:
 final_rank  final_refresh_score       suggested_action                                                                                                                                                   final_reason_codes  impressions_90d  sessions_90d trend_direction
          1            81.928467 refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate            12834            66            down
          2            81.728449 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate             8064            23            down
          3            81.639118 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|mo

## 7. Artifacts, reproducibility, and credits

The paper should embed the most interpretable artifacts from the workflow: a model-vs-baseline comparison table, a ranked queue preview, a feature-importance chart, and a summary of recommended action types. These outputs are already generated in the repo and can be pulled into a static article or GitHub Pages landing page.

### Reproducibility
This analysis is fully reproducible from the repo: the pipeline is run through `scripts/run_all.py`, which prepares the features, builds the baseline, trains the models, evaluates them, and exports the final queue and report. The outputs are stored in `outputs/` and can be used as the evidence base for the published paper.

### Acknowledgments & data credit
Built on the FlyRank ML Internship dataset. Source credit: https://flyrank.ai


In [ ]:
from pathlib import Path

output_root = Path("outputs")
print("Generated artifacts:")
for path in sorted(output_root.rglob("*")):
    if path.is_file():
        print(f"- {path.relative_to(Path('.'))}")
print("\nPipeline entry point:")
print("python scripts/run_all.py")
print("\nPublic-safe caveat:")
print("No client names, URLs, domains, raw queries, or credentials are included in the repo artifacts.")


Generated artifacts:

Pipeline entry point:
python scripts/run_all.py

Public-safe caveat:
No client names, URLs, domains, raw queries, or credentials are included in the repo artifacts.


## Self-check

Before you submit, confirm each line honestly:

- [x] The paper has a clear title and abstract at the top.
- [x] The question and decision are stated in plain language.
- [x] The data section explains the release, the public-safe scope, and the exclusions.
- [x] The methodology explains the target, the baseline, and the validation strategy.
- [x] The results section compares model vs baseline using the same split and honest numbers.
- [x] The limitations section is explicit about observed and directional claims.
- [x] The ranked recommendations section turns the model into an action playbook.
- [x] Reproducibility and source credit are included.
- [x] The paper uses careful language: observed, measured, directional, decision-support.
- [x] The notebook can be run top-to-bottom with no hidden assumptions about private data.
- [x] The final deployed page includes, at minimum: title + abstract, introduction, data, methodology, results, limitations, ranked recommendations, reproducibility, and acknowledgments.
- [x] The ML-12 demo / social / employer-facing summary is complete in the closing markdown cells.


## Week 8 showcase outline

### 5-minute demo outline
- 0:00-0:45 — Opening question: how can a content team decide which pages deserve a refresh review when they still attract some demand?
- 0:45-1:45 — Method: use an anonymized FlyRank-style content refresh dataset, train a ranked model, and compare it with a simple baseline on the same holdout split.
- 1:45-3:15 — One chart: show the top feature importance or the ranked queue signal that explains why the model favors some pages over others.
- 3:15-4:15 — One honest result: the random forest improved precision at 50 from 0.24 to 0.68, which is useful for prioritization rather than full automation.
- 4:15-5:00 — Recommendation: use the model as a reviewer aid for editorial triage, while keeping campaign-sensitive and policy-sensitive cases in human review.

### Short social post
I built a ranked refresh model to help content teams spot pages that may deserve review before they drift too far. The approach uses a simple, transparent modeling workflow and focuses on prioritization, not automation.

### Employer-facing summary
I built a ranking-based refresh model for a content-review use case using an anonymized FlyRank-style dataset. The model was trained to surface pages that still show demand but may be losing usefulness, and it outperformed a simple baseline on held-out data. The result was a practical triage tool for editorial review rather than an automated decision engine.
